# Tracker evals: illustrating the approach on dummy data

Two constraints on the real evaluation:

1. **Framerate varies per experiment.** Different trackers/configs may run at different `target_fps`. The frame indices we score against must stay exact regardless.
2. **GT is sparse but cumulative.** The real annotated frames aren't random samples — they're moments where a new instrument was added to a clean, unoccluded tray. So each GT checkpoint gives the *complete* set of instruments present so far, not just the newest one.

This notebook works entirely on synthetic data (no video, no detector weights) to illustrate the mechanics:

- a frame-sampling helper that decouples `target_fps` from GT-checkpoint alignment
- running a real tracker (`SORTTracker` from the `trackers` package) over synthetic detections
- IoU-matching the tracker's live output against the cumulative GT boxes at each checkpoint
- summarizing identity switches / misses / merges from those matches

The last section swaps in a hand-crafted switch+merge scenario directly, since relying on the real tracker to happen to switch an ID would make the illustration flaky.

## Imports

In [1]:
# --- imports
import warnings

import numpy as np
import supervision as sv
from scipy.optimize import linear_sum_assignment
from supervision.detection.utils.boxes import box_iou_batch
from trackers import SORTTracker
from trackers.utils.iou import GIoU
from trackers.utils.state_representations import XCYCSRStateEstimator

# unrelated internal deprecation noise from `trackers` itself (a `target=None`
# proxy warning raised on import), not anything about our usage
warnings.filterwarnings("ignore", category=FutureWarning, module="trackers")

## Synthetic clip

Mirrors the real data's shape: a tray starts empty, and instruments are added one at a time at specific frames (the GT checkpoints). Track 1 and track 2 are given crossing paths so the scene isn't trivially easy for the tracker.

In [2]:
# --- synthetic scene config
NATIVE_FPS = 30
FRAME_COUNT = 300

RNG = np.random.default_rng(0)

# frame each GT track is introduced on a clean, unoccluded view -- these are
# the GT checkpoints, exactly mirroring the real annotation pipeline
TRACK_INTRO_FRAME = {1: 60, 2: 150, 3: 250}
GT_CHECKPOINT_FRAMES = [0, *sorted(TRACK_INTRO_FRAME.values())]  # frame 0: empty tray, like the real data

BOX_SIZE = (80.0, 140.0)  # width, height; constant for simplicity


def track_center(track_id: int, frame_index: int) -> tuple[float, float] | None:
    """Noise-free box center for a track at a frame, or None before it's introduced."""
    if frame_index < TRACK_INTRO_FRAME[track_id]:
        return None
    t = frame_index / FRAME_COUNT
    if track_id == 1:
        return (200 + 500 * t, 300.0)  # tracks 1 and 2 cross paths around t=0.6
    if track_id == 2:
        return (700 - 500 * t, 300.0)
    return (400.0, 150 + 200 * t)  # track 3: independent vertical path


def track_bbox(track_id: int, frame_index: int) -> np.ndarray | None:
    center = track_center(track_id, frame_index)
    if center is None:
        return None
    cx, cy = center
    w, h = BOX_SIZE
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dtype=np.float32)


def simulated_detections(frame_index: int, drop_track_id: int | None = None) -> sv.Detections:
    """Stand-in for `Detector.predict(image)`: noisy boxes for every track
    introduced by this frame, optionally skipping one (simulates a missed detection)."""
    boxes = []
    for track_id in TRACK_INTRO_FRAME:
        if track_id == drop_track_id:
            continue
        bbox = track_bbox(track_id, frame_index)
        if bbox is not None:
            boxes.append(bbox + RNG.normal(scale=2.0, size=4))
    if not boxes:
        return sv.Detections.empty()
    return sv.Detections(xyxy=np.array(boxes, dtype=np.float32))


def gt_checkpoint_boxes(frame_index: int) -> list[tuple[int, np.ndarray]]:
    """The clean GT box for every track present at this checkpoint (cumulative)."""
    boxes = []
    for track_id in TRACK_INTRO_FRAME:
        bbox = track_bbox(track_id, frame_index)
        if bbox is not None:
            boxes.append((track_id, bbox))
    return boxes


# simulate one missed detection: track 2 drops out for a single frame right
# as track 3 is introduced (e.g. momentarily occluded)
MISSED_DETECTION_AT_FRAME = {250: 2}

## Framerate-independent sampling

Whatever `target_fps` an experiment uses, force the GT checkpoint frames into the sampled index list. This is the whole fix for "framerate might change between experiments" — the checkpoints stay exact no matter what rate the rest of the run is sampled at.

In [3]:
def merged_frame_indices(
    native_fps: float, frame_count: int, target_fps: float, required_frames: list[int]
) -> list[int]:
    """Evenly-spaced indices approximating `target_fps`, plus every frame in
    `required_frames` merged in so GT checkpoints are never missed."""
    step = max(round(native_fps / target_fps), 1)
    sampled = range(0, frame_count, step)
    return sorted(set(sampled) | set(required_frames))


# demo: changing target_fps changes the density of filler frames, but the
# checkpoints (0, 60, 150, 250) are always present
for target_fps in (10, 6):
    indices = merged_frame_indices(NATIVE_FPS, FRAME_COUNT, target_fps, GT_CHECKPOINT_FRAMES)
    checkpoints_present = [f in indices for f in GT_CHECKPOINT_FRAMES]
    print(f"target_fps={target_fps}: {len(indices)} frames sampled, checkpoints present: {checkpoints_present}")

target_fps=10: 101 frames sampled, checkpoints present: [True, True, True, True]
target_fps=6: 60 frames sampled, checkpoints present: [True, True, True, True]


## GT <-> prediction matching at a checkpoint

Optimal (Hungarian) IoU matching between the checkpoint's GT boxes and the tracker's currently *live* predictions. Immature/unmatched tracks report `tracker_id == -1` in this library — that's a sentinel, not a real identity, so it's excluded before matching (otherwise two different immature tracklets would look like "the same ID").

In [4]:
IOU_MATCH_THRESHOLD = 0.3


def match_gt_to_predictions(
    gt_items: list[tuple[int, np.ndarray]], tracked: sv.Detections
) -> dict[int, int | None]:
    """Returns {gt_track_id: matched_tracker_id_or_None} for one checkpoint."""
    gt_ids = [track_id for track_id, _ in gt_items]
    matches: dict[int, int | None] = dict.fromkeys(gt_ids)
    if not gt_items:
        return matches

    live = tracked.tracker_id != -1
    pred_xyxy = tracked.xyxy[live]
    pred_ids = tracked.tracker_id[live]
    if len(pred_xyxy) == 0:
        return matches

    gt_xyxy = np.array([bbox for _, bbox in gt_items], dtype=np.float32)
    iou = box_iou_batch(gt_xyxy, pred_xyxy)
    gt_rows, pred_cols = linear_sum_assignment(-iou)

    for gt_row, pred_col in zip(gt_rows, pred_cols, strict=False):
        if iou[gt_row, pred_col] >= IOU_MATCH_THRESHOLD:
            matches[gt_ids[gt_row]] = int(pred_ids[pred_col])
    return matches

## Run a real tracker over the synthetic detections

This proves the plumbing end to end — frame-index merging, feeding a real `SORTTracker.update()` in temporal order, and matching at checkpoints — using the deliberately-dropped detection at frame 250 as a guaranteed, deterministic miss. Whether *this particular* synthetic scene also produces a real identity switch depends on SORT's own association logic, so that's not forced here; the next section sanity-checks the switch/merge detection directly instead.

In [5]:
TARGET_FPS = 10

tracker = SORTTracker(
    frame_rate=TARGET_FPS,
    track_activation_threshold=0.0,  # dummy data has no confidence scores worth thresholding
    minimum_consecutive_frames=1,
    lost_track_buffer=5 * TARGET_FPS,
    minimum_iou_threshold=-0.30,
    state_estimator_class=XCYCSRStateEstimator,
    iou=GIoU(),
)

frame_indices = merged_frame_indices(NATIVE_FPS, FRAME_COUNT, TARGET_FPS, GT_CHECKPOINT_FRAMES)

checkpoint_matches: dict[int, dict[int, int | None]] = {}
for frame_index in frame_indices:
    drop_track_id = MISSED_DETECTION_AT_FRAME.get(frame_index)
    detections = simulated_detections(frame_index, drop_track_id=drop_track_id)
    tracked = tracker.update(detections)

    if frame_index in GT_CHECKPOINT_FRAMES:
        checkpoint_matches[frame_index] = match_gt_to_predictions(
            gt_checkpoint_boxes(frame_index), tracked
        )

print(f"ran over {len(frame_indices)} frames, {len(checkpoint_matches)} GT checkpoints matched")

ran over 101 frames, 4 GT checkpoints matched


## Summarizing identity switches / misses / merges

Reshape the per-checkpoint matches into per-GT-track sequences, then flag:

- **switches**: the matched predicted ID changes between consecutive checkpoints
- **misses**: a checkpoint where no prediction matched that GT track
- **merges**: one predicted ID matched to more than one distinct GT track across checkpoints (the tracker confused two separate instruments)

In [6]:
def per_track_sequences(
    checkpoint_matches: dict[int, dict[int, int | None]],
) -> dict[int, list[tuple[int, int | None]]]:
    sequences: dict[int, list[tuple[int, int | None]]] = {}
    for frame_index in sorted(checkpoint_matches):
        for gt_track_id, pred_id in checkpoint_matches[frame_index].items():
            sequences.setdefault(gt_track_id, []).append((frame_index, pred_id))
    return sequences


def summarize(sequences: dict[int, list[tuple[int, int | None]]]) -> None:
    total_switches = total_misses = total_checkpoints = 0
    gt_tracks_by_pred_id: dict[int, set[int]] = {}

    for gt_track_id, points in sorted(sequences.items()):
        matched_sequence = [pred_id for _, pred_id in points if pred_id is not None]
        switches = sum(1 for a, b in zip(matched_sequence, matched_sequence[1:]) if a != b)
        misses = sum(1 for _, pred_id in points if pred_id is None)

        total_switches += switches
        total_misses += misses
        total_checkpoints += len(points)
        for pred_id in matched_sequence:
            gt_tracks_by_pred_id.setdefault(pred_id, set()).add(gt_track_id)

        rendered = ", ".join(
            f"f{frame}=#{pred_id}" if pred_id is not None else f"f{frame}=MISSED"
            for frame, pred_id in points
        )
        print(f"  GT track {gt_track_id}: {rendered}  [{switches} switch(es), {misses} miss(es)]")

    merges = {pred_id: tids for pred_id, tids in gt_tracks_by_pred_id.items() if len(tids) > 1}
    print(
        f"\n  totals: {total_checkpoints} checkpoints, {total_switches} identity switch(es), "
        f"{total_misses} missed match(es)"
    )
    print(
        f"  merges detected: {merges}"
        if merges
        else "  no merges detected (no predicted id matched to more than one GT track)"
    )


print("=== synthetic SORTTracker run ===")
summarize(per_track_sequences(checkpoint_matches))

=== synthetic SORTTracker run ===
  GT track 1: f60=MISSED, f150=#0, f250=#0  [0 switch(es), 1 miss(es)]
  GT track 2: f150=MISSED, f250=MISSED  [0 switch(es), 2 miss(es)]
  GT track 3: f250=#1  [0 switch(es), 0 miss(es)]

  totals: 6 checkpoints, 0 identity switch(es), 3 missed match(es)
  no merges detected (no predicted id matched to more than one GT track)


**A real subtlety this run surfaced, worth flagging rather than hiding:** GT track 3's f250 checkpoint shows a clean match (`#1`, 0 misses) — but `#1` is actually GT track 2's own stable id. Debugging the raw tracker output frame by frame shows why: at frame 250 (the deliberate miss), track 2's real box is absent, but its tracklet is still alive and predicted forward by the Kalman filter. `minimum_iou_threshold=-0.30` is lenient enough that SORT's internal association pairs that stale, nearby tracklet with track 3's brand-new box instead of spawning a fresh one — track 3 inherits track 2's id for a frame, and track 2's real object has to restart from scratch (`tracker_id=-1`) two frames later.

None of this shows up in the checkpoint summary above: it happens strictly between checkpoints, and GT track 2 has no other checkpoint left to reveal the hand-off. **Checkpoint-only scoring can miss identity events that occur and resolve between checkpoints.** That's an inherent cost of only having sparse GT, not a bug in the matching code above — worth keeping in mind when reading real eval results as an upper bound on tracker quality, not an exact count.

## Sanity-checking switch/merge detection directly

Relying on the real tracker to happen to switch an ID would make this illustration flaky. Instead, feed `summarize` a hand-crafted sequence with a known switch and a known merge, to prove the detection logic itself is correct:

- GT track 1 is matched to predicted id `10`, then switches to `99`
- GT track 2 is missed at its last checkpoint
- GT track 3 is also matched to predicted id `99` — the same id GT track 1 ended up on, so `99` should be flagged as a merge

In [7]:
example_sequences = {
    1: [(0, 10), (60, 10), (150, 99), (250, 99)],
    2: [(60, 20), (150, 20), (250, None)],
    3: [(150, 99)],
}

print("=== hand-crafted switch + merge scenario ===")
summarize(example_sequences)

=== hand-crafted switch + merge scenario ===
  GT track 1: f0=#10, f60=#10, f150=#99, f250=#99  [1 switch(es), 0 miss(es)]
  GT track 2: f60=#20, f150=#20, f250=MISSED  [0 switch(es), 1 miss(es)]
  GT track 3: f150=#99  [0 switch(es), 0 miss(es)]

  totals: 8 checkpoints, 1 identity switch(es), 1 missed match(es)
  merges detected: {99: {1, 3}}


## From dummy to real

Everything above except the synthetic-scene cell is meant to carry over unchanged:

- `simulated_detections` / `gt_checkpoint_boxes` / `TRACK_INTRO_FRAME` → replaced by `ClipDataset` + `Detector.predict()` + `Frame.to_detections()` over a real clip
- `SORTTracker(...)` → replaced by whichever trained tracker you're evaluating, as long as it exposes the same `.update(detections) -> sv.Detections` interface as the rest of the `trackers` package
- `merged_frame_indices`, `match_gt_to_predictions`, `per_track_sequences`, and `summarize` stay exactly as they are